# System 1 Decisions - Sarvam Prism

In [1]:
from dotenv import load_dotenv
import os

## Setup API Keys

In [2]:
load_dotenv()
SARVAM_API_KEY = os.getenv('SARVAM_API_KEY')

assert SARVAM_API_KEY is not None, 'Could NOT load SARVAM_API_KEY from .env'

## Categorize transactions from Bank Statements

In [3]:
from sarvamai import SarvamAI

client = SarvamAI(
    api_subscription_key=SARVAM_API_KEY,
) 

API Interface similar to Jev (model from from TypeSafe AI)

In [4]:
response = client.decisions.create(
    model='sarvam-prism',
    state = { # A single transaction from a bank statement
        "date": "2026-10-10",
        "amount": 1276.0,
        "description": "Credit Interest Capitalised",
    },
    questions = { # include one or more questions here
        'categorize_transactions' : { # a single question
            'type': 'choice', # supported types: noul, choice and score
            'instructions': 'Which category best represents the bank transaction ?', # question body
            'criteria': { # possible answers with descriptions
                'INCOME_SALARY_RCVD' : 'Regular salary, freelancing, active income',
                'INVESTMENT_RETURNS_RCVD' : 'Interest, stock dividends, fund redemptions',
                'INTERNAL_TRANSFER_IN' : 'Moving money between your own accounts/cards',
                'REFUNDS_AND_REWARDS_RCVD' : 'Cashbacks, merchant refunds, tax returns',
                'MISC_INFLOW_RCVD' : 'Gifts from friends, unclassified incoming cash',
                'UTILITIES_AND_RENT_PAID' : 'Electricity, water, internet, phone, rent',
                'DEBT_AND_EMI_PAID' : 'Home loans, car loans, credit card minimums',
                'INSURANCE_AND_TAX_PAID': 'Medical/auto insurance, municipal/income taxes',
                'GROCERIES_AND_ESSENTIALS_PAID' : 'Supermarkets, local vendor produce, pharmacies',
                'LIFESTYLE_AND_DINING_PAID' : 'Restaurants, swiggy/zomato, movies, shopping, cafes',
                'TRAVEL_AND_TRANSPORT_PAID' : 'Fuel, Uber/Ola, train tickets, flights, auto repairs',
                'INTERNAL_TRANSFER_OUT' : 'Moving money out to your own investment/savings apps',
                'MISC_OUTFLOW_PAID' : 'True anomalies that dont fit anywhere else',                               
            },
            
        }
    }
)

Output probabilities are calibrated

95% probability means < 5% chance of going wrong

Claim: Steers away from highly confident wrong answers

In [5]:
print(response.model_dump_json(indent=2))

{
  "model": "sarvam-prism",
  "answers": {
    "categorize_transactions": {
      "type": "choice",
      "choice": "INVESTMENT_RETURNS_RCVD",
      "confidence": 0.9473865926265717,
      "probabilities": {
        "INCOME_SALARY_RCVD": 0.003945533651858568,
        "INVESTMENT_RETURNS_RCVD": 0.9514337778091431,
        "INTERNAL_TRANSFER_IN": 0.003701017936691642,
        "REFUNDS_AND_REWARDS_RCVD": 0.004457064904272556,
        "MISC_INFLOW_RCVD": 0.0077056135050952435,
        "UTILITIES_AND_RENT_PAID": 0.0033584320917725563,
        "DEBT_AND_EMI_PAID": 0.0042367842979729176,
        "INSURANCE_AND_TAX_PAID": 0.0034077558666467667,
        "GROCERIES_AND_ESSENTIALS_PAID": 0.0030763500835746527,
        "LIFESTYLE_AND_DINING_PAID": 0.003315458307042718,
        "TRAVEL_AND_TRANSPORT_PAID": 0.0031069901306182146,
        "INTERNAL_TRANSFER_OUT": 0.0036287829279899597,
        "MISC_OUTFLOW_PAID": 0.0046264007687568665
      }
    }
  },
  "usage": {
    "input_tokens": 381,
    "ou

Notice the following:
1. Does NOT output text, Outputs only probabilities
2. \# of Output Tokens is always zero
3. Billed for Input Tokens only
4. 4X to 15X less costly - Rs. 4 per million input tokens compared to Rs. 15 per million input tokens, Rs. 60 per million output tokens
5. Faster Responses compared to LLMs

## Categorizing multiple transactions (in a single API call)

In [6]:
transactions = [
    '30/06/2025	Credit Interest Capitalised 	1,177.00',
    '26/03/2026	1265	6.08555E+11	UPI-DR-608554609312-MOHAMMED HARIS J-BKID-806910110000396-taxi 	260',
    '31/03/2026	1265	1.20835E+11	UPI-DR-120835439135-AKSHAYAKALPA FARMS AND FOODS PRIVATE LIMITED-UTIB-1000129101195-WalletRecharge 	1,000.00',
    '31/03/2026	1265	1.20844E+11	UPI-DR-120843961332-PON STORE-YESB-001425000000051-butter milk '
]

In [7]:
instructions = 'Which category best represents this bank transaction ?'
criteria = {
    'INCOME_SALARY_RCVD' : 'Regular salary, freelancing, active income',
    'INVESTMENT_RETURNS_RCVD' : 'Interest, stock dividends, fund redemptions',
    'INTERNAL_TRANSFER_IN' : 'Moving money between your own accounts/cards',
    'REFUNDS_AND_REWARDS_RCVD' : 'Cashbacks, merchant refunds, tax returns',
    'MISC_INFLOW_RCVD' : 'Gifts from friends, unclassified incoming cash',
    'UTILITIES_AND_RENT_PAID' : 'Electricity, water, internet, phone, rent',
    'DEBT_AND_EMI_PAID' : 'Home loans, car loans, credit card minimums',
    'INSURANCE_AND_TAX_PAID': 'Medical/auto insurance, municipal/income taxes',
    'GROCERIES_AND_ESSENTIALS_PAID' : 'Supermarkets, local vendor produce, pharmacies',
    'LIFESTYLE_AND_DINING_PAID' : 'Restaurants, swiggy/zomato, movies, shopping, cafes',
    'TRAVEL_AND_TRANSPORT_PAID' : 'Fuel, Uber/Ola, train tickets, flights, auto repairs',
    'INTERNAL_TRANSFER_OUT' : 'Moving money out to your own investment/savings apps',
    'MISC_OUTFLOW_PAID' : 'True anomalies that dont fit anywhere else',                               
}

### Prepare questions

Each question is self-contained

In [8]:
import json

questions = {
    f'categorize_transaction_{i+1}' : {
        'type': 'choice', 
        'instructions': f'{instructions}:\n{txn}', 
        'criteria':  criteria
    } for i, txn in enumerate(transactions)
}

print(json.dumps(questions, indent=2))

{
  "categorize_transaction_1": {
    "type": "choice",
    "instructions": "Which category best represents this bank transaction ?:\n30/06/2025\tCredit Interest Capitalised \t1,177.00",
    "criteria": {
      "INCOME_SALARY_RCVD": "Regular salary, freelancing, active income",
      "INVESTMENT_RETURNS_RCVD": "Interest, stock dividends, fund redemptions",
      "INTERNAL_TRANSFER_IN": "Moving money between your own accounts/cards",
      "REFUNDS_AND_REWARDS_RCVD": "Cashbacks, merchant refunds, tax returns",
      "MISC_INFLOW_RCVD": "Gifts from friends, unclassified incoming cash",
      "UTILITIES_AND_RENT_PAID": "Electricity, water, internet, phone, rent",
      "DEBT_AND_EMI_PAID": "Home loans, car loans, credit card minimums",
      "INSURANCE_AND_TAX_PAID": "Medical/auto insurance, municipal/income taxes",
      "GROCERIES_AND_ESSENTIALS_PAID": "Supermarkets, local vendor produce, pharmacies",
      "LIFESTYLE_AND_DINING_PAID": "Restaurants, swiggy/zomato, movies, shopping, cafe

### Run questions throgh model

In [9]:
response = client.decisions.create(
    model='sarvam-prism',
    state = '', # state is empty since each question is self-contained
    questions = questions,
)

print(response.model_dump_json(indent=2))

{
  "model": "sarvam-prism",
  "answers": {
    "categorize_transaction_1": {
      "type": "choice",
      "choice": "INVESTMENT_RETURNS_RCVD",
      "confidence": 0.925588478644689,
      "probabilities": {
        "INCOME_SALARY_RCVD": 0.005397383589297533,
        "INVESTMENT_RETURNS_RCVD": 0.9313124418258667,
        "INTERNAL_TRANSFER_IN": 0.005647440440952778,
        "REFUNDS_AND_REWARDS_RCVD": 0.006802815943956375,
        "MISC_INFLOW_RCVD": 0.008625916205346584,
        "UTILITIES_AND_RENT_PAID": 0.004994773305952549,
        "DEBT_AND_EMI_PAID": 0.006081928964704275,
        "INSURANCE_AND_TAX_PAID": 0.004845719784498215,
        "GROCERIES_AND_ESSENTIALS_PAID": 0.0047525386326014996,
        "LIFESTYLE_AND_DINING_PAID": 0.004878511186689138,
        "TRAVEL_AND_TRANSPORT_PAID": 0.004756673704832792,
        "INTERNAL_TRANSFER_OUT": 0.00535587128251791,
        "MISC_OUTFLOW_PAID": 0.006548047531396151
      }
    },
    "categorize_transaction_2": {
      "type": "choice",

In [10]:
for i, txn in enumerate(transactions):
    question_id = f'categorize_transaction_{i+1}'
    assert question_id in questions
    assert question_id in response.answers

    best = response.answers[question_id].choice
    p = response.answers[question_id].probabilities[best]
    pct = int(100* round(p, 2))
    result = f"""
    {txn}
    Best Category: {best}
    Probability: {pct}%
    """.strip()
    
    print(result)
    print()

30/06/2025	Credit Interest Capitalised 	1,177.00
    Best Category: INVESTMENT_RETURNS_RCVD
    Probability: 93%

26/03/2026	1265	6.08555E+11	UPI-DR-608554609312-MOHAMMED HARIS J-BKID-806910110000396-taxi 	260
    Best Category: TRAVEL_AND_TRANSPORT_PAID
    Probability: 92%

31/03/2026	1265	1.20835E+11	UPI-DR-120835439135-AKSHAYAKALPA FARMS AND FOODS PRIVATE LIMITED-UTIB-1000129101195-WalletRecharge 	1,000.00
    Best Category: GROCERIES_AND_ESSENTIALS_PAID
    Probability: 39%

31/03/2026	1265	1.20844E+11	UPI-DR-120843961332-PON STORE-YESB-001425000000051-butter milk 
    Best Category: GROCERIES_AND_ESSENTIALS_PAID
    Probability: 76%

